In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.ensemble import RandomForestClassifier

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("customer_support.csv")

In [ ]:
print(df.shape)
print(df.head())
print(df.info())

(200000, 30)
   ticket_id      customer_name                   customer_email  \
0          1     Patricia Smith    patricia.smith760@outlook.com   
1          2  Patricia Williams   patricia.williams390@gmail.com   
2          3   William Anderson  william.anderson651@outlook.com   
3          4       David Miller       david.miller672@icloud.com   
4          5    Robert Gonzalez   robert.gonzalez391@hotmail.com   

           product                   category  \
0       Web Portal         Account Suspension   
1       Mobile App          Performance Issue   
2       Web Portal          Performance Issue   
3  Payment Gateway  Subscription Cancellation   
4       Web Portal            Feature Request   

                                   issue_description  \
0  The payment was deducted from my bank account ...   
1  I found a bug in the latest update affecting r...   
2  The application crashes whenever I try to uplo...   
3  My subscription was cancelled without my reque...   
4  

In [ ]:
df['escalated'] = df['escalated'].map({"Yes": 1, "No": 0, True: 1, False: 0})
df = df.dropna(subset=['escalated'])

In [ ]:
df['issue_description'] = df['issue_description'].fillna('')
df['text_length'] = df['issue_description'].apply(len)
df['urgent_flag'] = df['issue_description'].str.contains(
    "urgent|asap|immediately|angry|bad|error|fail", case=False
).astype(int)

In [ ]:
text_feature = 'issue_description'

categorical_features = [
    'product', 'category', 'priority', 'channel', 'region',
    'operating_system', 'browser', 'payment_method', 'language',
    'customer_segment'
]

numerical_features = [
    'customer_age', 'customer_tenure_months', 'previous_tickets',
    'customer_satisfaction_score', 'first_response_time_hours',
    'issue_complexity_score', 'text_length', 'urgent_flag'
]

X = df[categorical_features + numerical_features + [text_feature]]
y = df['escalated']


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
text_transformer = Pipeline(steps=[
    ('tfidf', TfidfVectorizer(
        max_features=20000,
        ngram_range=(1,2),
        stop_words='english'
    ))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('text', text_transformer, text_feature),
        ('cat', categorical_transformer, categorical_features),
        ('num', numerical_transformer, numerical_features)
    ]
)

In [ ]:
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=25,
    min_samples_split=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', model)
])


In [ ]:
clf.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('text',
                                                  Pipeline(steps=[('tfidf',
                                                                   TfidfVectorizer(max_features=20000,
                                                                                   ngram_range=(1,
                                                                                                2),
                                                                                   stop_words='english'))]),
                                                  'issue_description'),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['product', 'category',
                                                   'prio...
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['customer_age',
                                                   'customer_tenure_months',
                                                   'previous_tickets',
                                                   'customer_satisfaction_score',
                                                   'first_response_time_hours',
                                                   'issue_complexity_score',
                                                   'text_length',
                                                   'urgent_flag'])])),
                ('model',
                 RandomForestClassifier(class_weight='balanced', max_depth=25,
                                        min_samples_split=10, n_estimators=300,
                                        n_jobs=-1, random_state=42))])

In [ ]:
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

In [ ]:
print("Classification Report:\n")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:\n")
cm = confusion_matrix(y_test, y_pred)
print(cm)

print("ROC-AUC:", roc_auc_score(y_test, y_prob))


Classification Report:

              precision    recall  f1-score   support

           0       0.50      0.48      0.49     19916
           1       0.50      0.51      0.51     20084

    accuracy                           0.50     40000
   macro avg       0.50      0.50      0.50     40000
weighted avg       0.50      0.50      0.50     40000

Confusion Matrix:

[[ 9629 10287]
 [ 9803 10281]]
ROC-AUC: 0.4976270481411291


In [30]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=15,
        random_state=42,
        n_jobs=-1
    )
}

In [31]:
results = []

for name, model in models.items():
    clf = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred)
    })

In [32]:
results_df = pd.DataFrame(results)
print(results_df)

                 Model  Accuracy  Precision    Recall  F1 Score
0  Logistic Regression  0.499650   0.501489  0.586885  0.540837
1        Random Forest  0.501125   0.502994  0.539484  0.520601
